## Voting testes
Esse script implementa um ensemble por média entre dois modelos de Gradient Boosting:
* LightGBM (rodando em GPU, com categorias nativas)
* XGBoost (rodando em CPU, com árvores histogram)

## 1.Bibliotecas

In [9]:
import sys
import warnings
import os
import pandas as pd
import numpy as np
import joblib
import time
from pathlib import Path

# Modelagem e Métricas
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from sklearn.linear_model import LogisticRegression


from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             average_precision_score,roc_curve)



from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.ensemble import VotingClassifier

# Importações locais
from pathlib import Path
from setup_notebook import setup_path
setup_path()
from src.model_utils import *
from src.plot_metrica_class import *
from src.preprocess_utils_diab3a import * #(NOVO atualizações)


# from src.plot_metrica_class import * warnings.filterwarnings("ignore")
print(f"# Processo iniciado em: {time.strftime('%H:%M:%S')}")
start_inicial = time.time()

# Processo iniciado em: 19:08:39


## 2-DataLoad

In [2]:
# Definindo caminhos de pastas
BASE = Path.cwd().parent   
DATA_DIR = BASE / "data" / "raw"
DATA_MODELS = BASE / "models" 


# 1. Carregamento dos dados
# modo 1: usa todos os dados
dfo = pd.read_csv("/home/akel/PycharmProjects/Kaggle/Diabetes_Prediction_Challenge/data/raw/train.csv")
df = dfo.drop(columns='id')
TARGET = "diagnosed_diabetes"
X_train = df.drop(columns=TARGET)
y_train = df[TARGET]

# Dados de Teste (Kaggle)
df_test = pd.read_csv(DATA_DIR / "test.csv")
id_test = df_test["id"]
X_test_final = df_test.drop(columns='id')

# modo2:
# DATA_DIR = BASE / "data" / "raw"
# X_train = pd.read_csv(DATA_DIR / "X_train_raw.csv").reset_index(drop=True)
# X_val  = pd.read_csv(DATA_DIR / "X_test_raw.csv")
# y_train = pd.read_csv(DATA_DIR / "y_train_raw.csv").values.ravel()
# y_val  = pd.read_csv(DATA_DIR / "y_test_raw.csv")



# 2. Carregamento dos modelos pré-treinados
# Nota: Esses modelos geralmente são Pipelines que já incluem o preprocessador
pipe_XGB = joblib.load(DATA_MODELS / 'modelo_XGB2_final_randsearch.roc_auc_v3.joblib') # AUC Score : 0.7272
pipe_SVM = joblib.load(DATA_MODELS /'modelo_LinearSVC_final.roc_auc_v3a.joblib') # AUC Score 0.6951
search_h = joblib.load(DATA_MODELS /'search_Hrandomearly_stop_v3a.joblib') # AUC Score 0.7255
pipe_LGBM= search_h.best_estimator_
mtd_scoring='roc_auc'


print("✅ Dados e modelos carregados com sucesso!")

✅ Dados e modelos carregados com sucesso!


In [3]:
print("#Processo iniciado em:", time.strftime("%H:%M:%S"))

# 1. Definir os modelos e seus nomes
estimators = [
    ('xgb', pipe_XGB),
    ('lgbm', pipe_LGBM),
    ('svm', pipe_SVM)
]

# 2. Criar o Votador com Pesos Otimizados
# Sugestão de pesos: 2 para os fortes (0.72) e 1 para o estabilizador (0.69)
voter = VotingClassifier(
    estimators=estimators,
    voting='soft',
    weights=[2, 2, 1], # [XGB, LGBM, SVM]
    n_jobs=-1 # Treina/Executa em paralelo
)

print("# Iniciando o treinamento do Voting Ensemble...")
voter.fit(X_train, y_train)
print("✅ Voting Ensemble treinado!")
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))


#Processo iniciado em: 18:50:54
# Iniciando o treinamento do Voting Ensemble...
✅ Voting Ensemble treinado!

#Processo finalizado em: 18:53:37


In [18]:

# print("#Processo iniciado em:", time.strftime("%H:%M:%S"))

# # =====================================================
# # 1) Predição com o Voter
# # =====================================================
# # O voter.predict utiliza os pesos definidos para a classe final
# y_pred_voter = voter.predict(X_val)

# # =====================================================
# # 2) Otimização do threshold de decisão
# # =====================================================
# # Aqui usamos o voter para encontrar o threshold que melhor equilibra Precision/Recall
# best_th_voter, max_acc_voter, y_probs_voter = best_threshold2(voter, X_train, y_train, X_val, y_val)

# # =====================================================
# # 3) Desempenho em validação cruzada (Opcional, mas lento)
# # =====================================================
# # Nota: Isso treinará o ensemble 3 vezes. Se estiver com pressa, pode pular esta parte.
# cv_s = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
# cv_scores_voter = cross_val_score(voter, X_train, y_train,
#                                  cv=cv_s,
#                                  scoring=mtd_scoring,
#                                  n_jobs=-1)

# print(f"\n{'='*70}")
# print(f" 🏆 RESULTADOS DO VOTING ENSEMBLE (XGB + LGBM + SVM) ".center(70))
# print(f"{'='*70}")

# # =====================================================
# # 4) Avaliação por validação cruzada (Treino)
# # =====================================================
# print("📊 CROSS-VALIDATION")
# print(f"   Média {mtd_scoring}:       {cv_scores_voter.mean():>15.4f} ± {cv_scores_voter.std():.4f}")

# # =====================================================
# # 5) Avaliação no conjunto de teste (Métricas Principais)
# # =====================================================
# print(f"\n✅ TEST SET PERFORMANCE")
# print(f"   Acurácia Padrão (0.5):      {accuracy_score(y_val, y_pred_voter):>10.4f}")
# print(f"   Acurácia Otimizada:         {max_acc_voter:>10.4f} (threshold ={best_th_voter:>6.3f})")
# print(f"   ROC-AUC:                    {roc_auc_score(y_val, y_probs_voter):>10.4f}")
# print(f"   Avg Precision (PRC):        {average_precision_score(y_val, y_probs_voter):>10.4f}")

# # =====================================================
# # 6) Detalhamento: Precision, Recall e F1-Score
# # =====================================================
# # Essencial para ver se o ensemble melhorou a detecção de diabetes (Classe 1)
# print(f"\n📝 RELATÓRIO DE CLASSIFICAÇÃO (Threshold 0.5):")
# print(classification_report(y_val, y_pred_voter))

# # =====================================================
# # 7) Composição do Ensemble
# # =====================================================
# print(f"{'='*70}")
# print("🎯 ESTRUTURA DO ENSEMBLE")
# for name, weight in zip([e[0] for e in voter.estimators], voter.weights):
#     print(f" • Modelo: {name.upper():<10} | Peso: {weight}")

# print(f"{'='*70}")
# print(f"Processo finalizado em: {time.strftime('%H:%M:%S')}")

In [12]:
preds = pd.DataFrame({
    'xgb': pipe_XGB.predict_proba(X_val)[:,1],
    'lgbm': pipe_LGBM.predict_proba(X_val)[:,1],
    'svm': pipe_SVM.predict_proba(X_val)[:,1]
})
print(preds.corr())

           xgb      lgbm       svm
xgb   1.000000  0.989427  0.877387
lgbm  0.989427  1.000000  0.881261
svm   0.877387  0.881261  1.000000


In [17]:
# Testando a Estratégia A (Mais diversidade)
pesos_A = [1, 1, 1.5]
soma_pesos_A = sum(pesos_A)
prob_final_A = (preds['xgb']*pesos_A[0] + preds['lgbm']*pesos_A[1] + preds['svm']*pesos_A[2]) / soma_pesos_A

# Testando a Estratégia B (Mais performance individual)
pesos_B = [1.5, 1.5, 1.0]
soma_pesos_B = sum(pesos_B)
prob_final_B = (preds['xgb']*pesos_B[0] + preds['lgbm']*pesos_B[1] + preds['svm']*pesos_B[2]) / soma_pesos_B

# Compare o ROC-AUC de cada uma
from sklearn.metrics import roc_auc_score
print(f"AUC Estratégia A: {roc_auc_score(y_val, prob_final_A):.5f}")
print(f"AUC Estratégia B: {roc_auc_score(y_val, prob_final_B):.5f}")


# Estratégia C: Ponderação pelo AUC Individual (proporcional ao que cada um entregou)
pesos_C = [0.7272, 0.7255, 0.6951] 
soma_C = sum(pesos_C)
prob_final_C = (preds['xgb']*pesos_C[0] + preds['lgbm']*pesos_C[1] + preds['svm']*pesos_C[2]) / soma_C

print(f"AUC Estratégia C: {roc_auc_score(y_val, prob_final_C):.5f}")
# Estratégia D: Apenas os Gigantes (Sem o SVM)
# Como são quase iguais, peso 1 para cada
prob_final_D = (preds['xgb'] + preds['lgbm']) / 2

print(f"AUC Estratégia D (XGB + LGBM): {roc_auc_score(y_val, prob_final_D):.5f}")

AUC Estratégia A: 0.72225
AUC Estratégia B: 0.72588
AUC Estratégia C: 0.72463
AUC Estratégia D (XGB + LGBM): 0.72811


In [ ]:
# 1. Gerar as probabilidades no conjunto de teste final (o do Kaggle)
print("🚀 Gerando predições finais com o Power Duo (XGB + LGBM)...")

prob_xgb_final = pipe_XGB.predict_proba(X_test_final)[:, 1]
prob_lgbm_final = pipe_LGBM.predict_proba(X_test_final)[:, 1]

# 2. Média simples (Estratégia D)
prob_voting_final = (prob_xgb_final + prob_lgbm_final) / 2

# 3. Criar o DataFrame de submissão
submission = pd.DataFrame({
    "id": id_test,
    "diagnosed_diabetes": prob_voting_final
})

# 4. Salvar o arquivo
filename = "submission_final_voting_XGB_LGBM_v1.csv"
submission.to_csv(filename, index=False)

print(f"✅ Arquivo '{filename}' gerado com sucesso!")